In [0]:
import subprocess

subprocess.run([
    "pip", "install",
    "azure-storage-blob",
    "pyarrow",
    "pandas",
    "-q"
])

CompletedProcess(args=['pip', 'install', 'azure-storage-blob', 'pyarrow', 'pandas', '-q'], returncode=0)

In [0]:
from pyspark.sql import functions as F

spark.conf.set("spark.sql.session.timeZone", "America/New_York")
print("Spark version:", spark.version)

ACCOUNT_NAME = "nyctrafficlakehouse"
CONTAINER = "nyc-traffic-lakehouse"
ACCOUNT_KEY = "muon xem khong"

# Dùng biến này cho MỌI lệnh đọc/ghi wasbs:// (Databricks Serverless cần khai báo credential riêng mỗi lệnh)
AZURE_OPT_KEY = f"fs.azure.account.key.{ACCOUNT_NAME}.blob.core.windows.net"

BASE_PATH = f"wasbs://{CONTAINER}@{ACCOUNT_NAME}.blob.core.windows.net"
GOLD_PATH = f"{BASE_PATH}/gold/training_features"
MODEL_PATH = f"{BASE_PATH}/artifacts/model/random_forest_model"

print("GOLD_PATH :", GOLD_PATH)
print("MODEL_PATH:", MODEL_PATH)

Spark version: 4.1.0
GOLD_PATH : wasbs://nyc-traffic-lakehouse@nyctrafficlakehouse.blob.core.windows.net/gold/training_features
MODEL_PATH: wasbs://nyc-traffic-lakehouse@nyctrafficlakehouse.blob.core.windows.net/artifacts/model/random_forest_model


In [0]:
from pyspark.ml.feature import StringIndexer, VectorAssembler
from pyspark.ml.classification import RandomForestClassifier

idx = StringIndexer(inputCol="link_id", outputCol="link_id_idx", handleInvalid="keep")
va = VectorAssembler(inputCols=["hour"], outputCol="features")

print("MLlib OK")

MLlib OK


In [0]:
df = (spark.read
    .format("delta")
    .option(AZURE_OPT_KEY, ACCOUNT_KEY)
    .load(GOLD_PATH))

print(f"Tổng rows Gold: {df.count():,}")

label_col = "future_congestion_15min"

feature_cols = [
    "link_id", "borough",
    "hour", "day_of_week", "month", "is_weekend", "is_holiday",
    "speed_ratio", "current_congestion",
    "past_congestion_15min", "past_speed_ratio_15min",
    "past_congestion_30min", "past_speed_ratio_30min",
    "past_congestion_45min", "past_speed_ratio_45min",
    "past_congestion_60min", "past_speed_ratio_60min",
    "congestion_trend", "speed_ratio_trend"
]

required_cols = list(dict.fromkeys(feature_cols + [label_col, "data_as_of", "year"]))
missing_cols = [c for c in required_cols if c not in df.columns]
assert len(missing_cols) == 0, f"Thiếu cột trong Gold: {missing_cols}"

print("Số feature:", len(feature_cols))
print("Required cols:", required_cols)

Tổng rows Gold: 21,887,219
Số feature: 19
Required cols: ['link_id', 'borough', 'hour', 'day_of_week', 'month', 'is_weekend', 'is_holiday', 'speed_ratio', 'current_congestion', 'past_congestion_15min', 'past_speed_ratio_15min', 'past_congestion_30min', 'past_speed_ratio_30min', 'past_congestion_45min', 'past_speed_ratio_45min', 'past_congestion_60min', 'past_speed_ratio_60min', 'congestion_trend', 'speed_ratio_trend', 'future_congestion_15min', 'data_as_of', 'year']


In [0]:
df_model_raw = df.select(*required_cols)

df_model_raw = (
    df_model_raw
    .withColumn("link_id", F.col("link_id").cast("string"))
    .withColumn("borough", F.col("borough").cast("string"))
    .withColumn(label_col, F.col(label_col).cast("double"))
)

numeric_cast_cols = [
    "hour", "day_of_week", "month", "is_weekend", "is_holiday",
    "speed_ratio", "current_congestion",
    "past_congestion_15min", "past_speed_ratio_15min",
    "past_congestion_30min", "past_speed_ratio_30min",
    "past_congestion_45min", "past_speed_ratio_45min",
    "past_congestion_60min", "past_speed_ratio_60min",
    "congestion_trend", "speed_ratio_trend",
    "year"
]
for c in numeric_cast_cols:
    df_model_raw = df_model_raw.withColumn(c, F.col(c).cast("double"))

df_model_raw = df_model_raw.dropna(subset=feature_cols + [label_col, "year", "month"])

# SAMPLE 25% TOÀN BỘ dataset NGAY TỪ ĐẦU, giữ tỉ lệ 3 class
SAMPLE_FRACTION = 0.25
fractions = {0.0: SAMPLE_FRACTION, 1.0: SAMPLE_FRACTION, 2.0: SAMPLE_FRACTION}

df_model_sampled = df_model_raw.sampleBy(label_col, fractions=fractions, seed=42)

# Checkpoint Delta (thay cho .cache(), vì Databricks Serverless không hỗ trợ .cache())
TMP_MODEL_CHECKPOINT = f"{BASE_PATH}/tmp/df_model_small_checkpoint"

(df_model_sampled.write
    .format("delta")
    .option(AZURE_OPT_KEY, ACCOUNT_KEY)
    .mode("overwrite")
    .save(TMP_MODEL_CHECKPOINT))

df_model = (spark.read
    .format("delta")
    .option(AZURE_OPT_KEY, ACCOUNT_KEY)
    .load(TMP_MODEL_CHECKPOINT))

n_model = df_model.count()
print(f"Rows sau sample {SAMPLE_FRACTION*100:.0f}%: {n_model:,}")
df_model.groupBy(label_col).count().orderBy(label_col).show()

Rows sau sample 25%: 5,475,042
+-----------------------+-------+
|future_congestion_15min|  count|
+-----------------------+-------+
|                    0.0|3752732|
|                    1.0| 806372|
|                    2.0| 915938|
+-----------------------+-------+



In [0]:
df_train = df_model.filter(
    (F.col("year") == 2024) | ((F.col("year") == 2025) & (F.col("month") <= 9))
)
df_val = df_model.filter(
    (F.col("year") == 2025) & (F.col("month").between(10, 12))
)
df_test = df_model.filter(F.col("year") >= 2026)

train_rows = df_train.count()
val_rows = df_val.count()
test_rows = df_test.count()
print(f"Train rows: {train_rows:,}")
print(f"Val rows  : {val_rows:,}")
print(f"Test rows : {test_rows:,}")

print("Train distribution:")
df_train.groupBy(label_col).count().orderBy(label_col).show()
print("Val distribution:")
df_val.groupBy(label_col).count().orderBy(label_col).show()
print("Test distribution:")
df_test.groupBy(label_col).count().orderBy(label_col).show()

Train rows: 3,951,239
Val rows  : 568,350
Test rows : 954,281
Train distribution:
+-----------------------+-------+
|future_congestion_15min|  count|
+-----------------------+-------+
|                    0.0|2732854|
|                    1.0| 579208|
|                    2.0| 639177|
+-----------------------+-------+

Val distribution:
+-----------------------+------+
|future_congestion_15min| count|
+-----------------------+------+
|                    0.0|388405|
|                    1.0| 82198|
|                    2.0| 97747|
+-----------------------+------+

Test distribution:
+-----------------------+------+
|future_congestion_15min| count|
+-----------------------+------+
|                    0.0|630562|
|                    1.0|144892|
|                    2.0|178827|
+-----------------------+------+



In [0]:
num_classes = 3
class_counts = df_train.groupBy(label_col).count()
total_train = df_train.count()

weights_df = (
    class_counts
    .withColumn("class_weight", F.lit(total_train) / (F.lit(num_classes) * F.col("count")))
    .select(F.col(label_col).alias("label_for_weight"), F.col("class_weight").cast("double"))
)
print("Class weight tính trên Train:")
weights_df.orderBy("label_for_weight").show()

df_train_w = (
    df_train
    .join(F.broadcast(weights_df), df_train[label_col] == weights_df["label_for_weight"], "left")
    .drop("label_for_weight")
)

Class weight tính trên Train:
+----------------+------------------+
|label_for_weight|      class_weight|
+----------------+------------------+
|             0.0|0.4819429309676502|
|             1.0| 2.273932104989342|
|             2.0|2.0605867649597323|
+----------------+------------------+



In [0]:
from pyspark.ml import Pipeline
from pyspark.ml.feature import StringIndexer, VectorAssembler
from pyspark.ml.classification import RandomForestClassifier

categorical_indexers = [
    StringIndexer(inputCol="link_id", outputCol="link_id_idx", handleInvalid="keep"),
    StringIndexer(inputCol="borough", outputCol="borough_idx", handleInvalid="keep"),
]

numeric_features = [
    "hour", "day_of_week", "month", "is_weekend", "is_holiday",
    "speed_ratio", "current_congestion",
    "past_congestion_15min", "past_speed_ratio_15min",
    "past_congestion_30min", "past_speed_ratio_30min",
    "past_congestion_45min", "past_speed_ratio_45min",
    "past_congestion_60min", "past_speed_ratio_60min",
    "congestion_trend", "speed_ratio_trend",
]

assembler = VectorAssembler(
    inputCols=["link_id_idx", "borough_idx"] + numeric_features,
    outputCol="features",
    handleInvalid="keep"
)

# === Params cố định — model nhỏ ===
FINAL_PARAMS = {
    "numTrees": 80,
    "maxDepth": 13,
    "minInstancesPerNode": 5,
}

rf = RandomForestClassifier(
    featuresCol="features",
    labelCol=label_col,
    weightCol="class_weight",
    predictionCol="prediction",
    probabilityCol="probability",
    rawPredictionCol="rawPrediction",
    numTrees=FINAL_PARAMS["numTrees"],
    maxDepth=FINAL_PARAMS["maxDepth"],
    minInstancesPerNode=FINAL_PARAMS["minInstancesPerNode"],
    maxBins=132,
    impurity="gini",
    featureSubsetStrategy="sqrt",
    seed=42,
    subsamplingRate=0.8,
    maxMemoryInMB=256,
)

pipeline = Pipeline(stages=categorical_indexers + [assembler, rf])
print("Pipeline + params cố định:", FINAL_PARAMS)

Pipeline + params cố định: {'numTrees': 80, 'maxDepth': 13, 'minInstancesPerNode': 5}


In [0]:
import time

t0 = time.time()
model = pipeline.fit(df_train_w)
fit_time = time.time() - t0
print(f" Fit xong sau {fit_time/60:.2f} phút")

In [0]:
def compute_macro_f1_fast(pred_df, label_col=label_col, prediction_col="prediction"):
    cm_rows = pred_df.groupBy(label_col, prediction_col).count().collect()
    classes = [0.0, 1.0, 2.0]
    cm = {}
    for r in cm_rows:
        cm[(float(r[label_col]), float(r[prediction_col]))] = int(r["count"])

    total_rows = sum(cm.values())
    metrics = []
    for c in classes:
        tp = cm.get((c, c), 0)
        fp = sum(cm.get((other, c), 0) for other in classes if other != c)
        fn = sum(cm.get((c, other), 0) for other in classes if other != c)
        precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
        recall = tp / (tp + fn) if (tp + fn) > 0 else 0.0
        f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0.0
        metrics.append((c, tp, fp, fn, precision, recall, f1))

    macro_f1 = sum(row[-1] for row in metrics) / len(classes)
    metrics_df = spark.createDataFrame(metrics, ["class", "tp", "fp", "fn", "precision", "recall", "f1"])
    return macro_f1, metrics_df, total_rows

In [0]:
pred_val = model.transform(df_val).select(label_col, "prediction")
macro_f1_val, metrics_val_df, n_val_pred = compute_macro_f1_fast(pred_val)

print(f"Val rows: {n_val_pred:,}")
print(f"Macro F1 (Val): {macro_f1_val:.6f}")
metrics_val_df.orderBy("class").show()

In [0]:
pred_test = model.transform(df_test).select(label_col, "prediction")
macro_f1_test, metrics_test_df, n_test_pred = compute_macro_f1_fast(pred_test)

print(f"Test rows: {n_test_pred:,}")
print(f"Macro F1 (Test): {macro_f1_test:.6f}")
metrics_test_df.orderBy("class").show()

In [0]:
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import pandas as pd

# Lấy confusion matrix dạng đầy đủ (3x3) từ predictions Test
cm_pdf = (pred_test
    .groupBy(label_col, "prediction")
    .count()
    .toPandas())

classes = [0.0, 1.0, 2.0]
class_names = ["Thông thoáng (0)", "Tắc vừa (1)", "Tắc nặng (2)"]

cm_matrix = np.zeros((3, 3))
for _, row in cm_pdf.iterrows():
    true_idx = classes.index(row[label_col])
    pred_idx = classes.index(row["prediction"])
    cm_matrix[true_idx, pred_idx] = row["count"]

fig, ax = plt.subplots(figsize=(7, 6))
sns.heatmap(
    cm_matrix.astype(int),
    annot=True, fmt="d", cmap="Blues",
    xticklabels=class_names, yticklabels=class_names,
    cbar_kws={"label": "Số lượng"},
    ax=ax
)
ax.set_xlabel("Predicted")
ax.set_ylabel("Actual")
ax.set_title(f"Confusion Matrix — Test Set (Macro F1 = {macro_f1_test:.4f})")
plt.tight_layout()
plt.show()

In [0]:
cm_normalized = cm_matrix / cm_matrix.sum(axis=1, keepdims=True)

fig, ax = plt.subplots(figsize=(7, 6))
sns.heatmap(
    cm_normalized,
    annot=True, fmt=".1%", cmap="Oranges",
    xticklabels=class_names, yticklabels=class_names,
    cbar_kws={"label": "% theo Actual"},
    ax=ax
)
ax.set_xlabel("Predicted")
ax.set_ylabel("Actual")
ax.set_title("Confusion Matrix (chuẩn hóa theo %) — Test Set")
plt.tight_layout()
plt.show()

In [0]:
metrics_pdf = metrics_test_df.orderBy("class").toPandas()
metrics_pdf["class_name"] = class_names

fig, ax = plt.subplots(figsize=(9, 5))
x = np.arange(len(class_names))
width = 0.25

ax.bar(x - width, metrics_pdf["precision"], width, label="Precision", color="#4C72B0")
ax.bar(x,         metrics_pdf["recall"],    width, label="Recall",    color="#DD8452")
ax.bar(x + width, metrics_pdf["f1"],        width, label="F1",        color="#55A868")

ax.set_xticks(x)
ax.set_xticklabels(class_names)
ax.set_ylim(0, 1)
ax.set_ylabel("Score")
ax.set_title("Precision / Recall / F1 theo từng class — Test Set")
ax.legend()
ax.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.show()

Bắt đầu lưu model

In [0]:
spark.sql("SHOW CATALOGS").show(truncate=False)

In [0]:
spark.sql("SHOW SCHEMAS IN nyc_traffic_databricks").show(truncate=False)


In [0]:
spark.sql("CREATE VOLUME IF NOT EXISTS nyc_traffic_databricks.default.mlflow_tmp")

In [0]:
import mlflow
import mlflow.spark

UC_VOLUME_TMP = "/Volumes/nyc_traffic_databricks/default/mlflow_tmp"

with mlflow.start_run(run_name="rf_congestion_model_small") as run:
    mlflow.log_params(FINAL_PARAMS)
    mlflow.log_param("sample_fraction", SAMPLE_FRACTION)
    mlflow.log_metric("macro_f1_val", macro_f1_val)
    mlflow.log_metric("macro_f1_test", macro_f1_test)

    mlflow.spark.log_model(
        model,
        artifact_path="random_forest_model",
        dfs_tmpdir=UC_VOLUME_TMP
    )

    run_id = run.info.run_id
    print(f"✅ Đã log model. Run ID: {run_id}")

In [0]:
# Xóa sạch model cũ trên Azure trước khi upload bản mới — tránh rác do hash stage khác nhau giữa các lần fit
container_client = blob_service.get_container_client(CONTAINER)
old_blobs = list(container_client.list_blobs(name_starts_with=BLOB_MODEL_PREFIX))
for b in old_blobs:
    container_client.delete_blob(b.name)
print(f"Đã xóa {len(old_blobs)} file model cũ trước khi upload bản mới")

In [0]:
import os
from azure.storage.blob import BlobServiceClient

# Download từ MLflow artifact storage về local disk của cluster
local_download_path = mlflow.artifacts.download_artifacts(
    artifact_uri=f"runs:/{run_id}/random_forest_model"
)
print(f"Đã download model về: {local_download_path}")

# Upload toàn bộ thư mục đó lên Azure Blob, đúng path doc yêu cầu
CONN_STR = f"DefaultEndpointsProtocol=https;AccountName={ACCOUNT_NAME};AccountKey={ACCOUNT_KEY};EndpointSuffix=core.windows.net"
blob_service = BlobServiceClient.from_connection_string(CONN_STR)

BLOB_MODEL_PREFIX = "artifacts/model/random_forest_model"

uploaded_count = 0
for root, dirs, files in os.walk(local_download_path):
    for fname in files:
        local_file = os.path.join(root, fname)
        rel_path = os.path.relpath(local_file, local_download_path)
        blob_path = f"{BLOB_MODEL_PREFIX}/{rel_path}".replace("\\", "/")
        with open(local_file, "rb") as f:
            blob_service.get_blob_client(container=CONTAINER, blob=blob_path).upload_blob(f, overwrite=True)
        uploaded_count += 1

print(f" Đã upload {uploaded_count} file lên Azure Blob: {BLOB_MODEL_PREFIX}")
print(f"MODEL_PATH (đúng theo doc): {MODEL_PATH}")

Đã download model về: /local_disk0/user_tmp_data/spark-215e7ba1-a204-4c1b-a9a1-7d/tmpnnratf3s/random_forest_model
 Đã upload 44 file lên Azure Blob: artifacts/model/random_forest_model
MODEL_PATH (đúng theo doc): wasbs://nyc-traffic-lakehouse@nyctrafficlakehouse.blob.core.windows.net/artifacts/model/random_forest_model


In [0]:
import os, shutil
from pyspark.ml import PipelineModel

UC_VOLUME_TMP = "/Volumes/nyc_traffic_databricks/default/mlflow_tmp"
local_verify_path = f"{UC_VOLUME_TMP}/rf_model_verify_check"

# XÓA SẠCH trước khi download lại — tránh lẫn file cũ/hỏng
if os.path.exists(local_verify_path):
    shutil.rmtree(local_verify_path)
os.makedirs(local_verify_path, exist_ok=True)

container_client = blob_service.get_container_client(CONTAINER)
n_downloaded = 0
for blob in container_client.list_blobs(name_starts_with=BLOB_MODEL_PREFIX):
    rel_path = blob.name[len(BLOB_MODEL_PREFIX) + 1:]
    local_target = os.path.join(local_verify_path, rel_path)
    os.makedirs(os.path.dirname(local_target), exist_ok=True)

    blob_data = container_client.download_blob(blob.name).readall()
    # Check size > 0 để phát hiện sớm nếu blob nào đó rỗng/lỗi
    if len(blob_data) == 0:
        print(f"⚠️ CẢNH BÁO: blob rỗng — {blob.name}")

    with open(local_target, "wb") as f:
        f.write(blob_data)
    n_downloaded += 1

print(f"Đã download {n_downloaded} file để verify")

# Kiểm tra nhanh file metadata chính trước khi load (debug nếu còn lỗi)
metadata_path = os.path.join(local_verify_path, "metadata", "part-00000")
if os.path.exists(metadata_path):
    with open(metadata_path, "r") as f:
        content = f.read()
        print(f"metadata/part-00000 ({len(content)} chars): {content[:200]}")
else:
    print("⚠️ Không tìm thấy metadata/part-00000 — kiểm tra cấu trúc thư mục")
    for root, dirs, files in os.walk(local_verify_path):
        for fn in files:
            print(os.path.join(root, fn))

loaded_model = PipelineModel.load(f"{local_verify_path}/sparkml")
print("✅ Load model thành công")
print(loaded_model.stages)

⚠️ CẢNH BÁO: blob rỗng — artifacts/model/random_forest_model/sparkml/metadata/_SUCCESS
⚠️ CẢNH BÁO: blob rỗng — artifacts/model/random_forest_model/sparkml/metadata/_started_1494797227377650467
⚠️ CẢNH BÁO: blob rỗng — artifacts/model/random_forest_model/sparkml/metadata/_started_18084640773689659
⚠️ CẢNH BÁO: blob rỗng — artifacts/model/random_forest_model/sparkml/stages/0_StringIndexer_0ae725a7ff90/data/_SUCCESS
⚠️ CẢNH BÁO: blob rỗng — artifacts/model/random_forest_model/sparkml/stages/0_StringIndexer_0ae725a7ff90/data/_started_6273225861910216243
⚠️ CẢNH BÁO: blob rỗng — artifacts/model/random_forest_model/sparkml/stages/0_StringIndexer_0ae725a7ff90/metadata/_SUCCESS
⚠️ CẢNH BÁO: blob rỗng — artifacts/model/random_forest_model/sparkml/stages/0_StringIndexer_0ae725a7ff90/metadata/_started_2503839993283989636
⚠️ CẢNH BÁO: blob rỗng — artifacts/model/random_forest_model/sparkml/stages/0_StringIndexer_b5906fd611eb/data/_SUCCESS
⚠️ CẢNH BÁO: blob rỗng — artifacts/model/random_forest_mod

In [0]:
quick_val = df_val.select(*(feature_cols + [label_col])).limit(20)
quick_pred = loaded_model.transform(quick_val).select(
    "link_id", "borough", label_col, "prediction", "probability"
)
quick_pred.show(20, truncate=False)

+-------+-------+-----------------------+----------+--------------------------------------------------------------+
|link_id|borough|future_congestion_15min|prediction|probability                                                   |
+-------+-------+-----------------------+----------+--------------------------------------------------------------+
|4362251|Queens |0.0                    |0.0       |[0.9404054850287459,0.04583517573108743,0.013759339240166693] |
|4362251|Queens |0.0                    |0.0       |[0.9404054850287459,0.04583517573108743,0.013759339240166693] |
|4616261|Bronx  |0.0                    |1.0       |[0.45346407194352106,0.4959991841346699,0.050536743921808965] |
|4616261|Bronx  |0.0                    |0.0       |[0.7440262376306501,0.22381996100479068,0.03215380136455922]  |
|4616261|Bronx  |0.0                    |1.0       |[0.27258176573882464,0.6464775400452837,0.08094069421589171]  |
|4616261|Bronx  |0.0                    |1.0       |[0.4543051636336197,

In [0]:
print("=== VERIFY 4 ARTIFACTS TRÊN AZURE BLOB ===\n")

artifacts_to_check = [
    "artifacts/model/random_forest_model/",
    "artifacts/free_flow_speed_lookup.parquet",
    "artifacts/active_link_ids.json",
    "artifacts/link_coordinates.parquet",
]

container_client = blob_service.get_container_client(CONTAINER)

for artifact in artifacts_to_check:
    blobs = list(container_client.list_blobs(name_starts_with=artifact))
    if len(blobs) > 0:
        total_size = sum(b.size for b in blobs)
        print(f"✅ {artifact} — {len(blobs)} file, tổng {total_size:,} bytes")
    else:
        print(f"❌ {artifact} — KHÔNG TÌM THẤY")

=== VERIFY 4 ARTIFACTS TRÊN AZURE BLOB ===

✅ artifacts/model/random_forest_model/ — 44 file, tổng 14,061,552 bytes
✅ artifacts/free_flow_speed_lookup.parquet — 3 file, tổng 5,299 bytes
✅ artifacts/active_link_ids.json — 1 file, tổng 1,375 bytes
✅ artifacts/link_coordinates.parquet — 95 file, tổng 97,779 bytes
